In [5]:
# load libraries 
import pandas as pd
import numpy as np
import re
import math
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path

In [6]:
# get the SUN covid study dataset
Covid51countries = pd.read_csv(Path("~/Projects/hypocognition/data/raw/Covid51countries.csv").expanduser())
Covid51countries

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,gender,ladder,employment,losejob,covidsymptom,language,datasource,ISO3,longstring,na_count
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,1.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,3,0
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,2.0,7.0,3,NaN,NaN,ZH-S,snowball,AUS,3,0
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,1.0,4.0,1,NaN,0.0,ZH-S,snowball,AUS,5,0
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,2.0,5.0,4,NaN,0.0,ZH-S,snowball,AUS,4,0
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,1.0,5.0,1,NaN,0.0,ZH-S,snowball,AUS,3,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24216,92,Kenya,25/04/2020 01:25,25/04/2020 02:44,0.0,100.0,4690.0,1.0,25/04/2020 02:44,100.0,...,2.0,4.0,4,NaN,0.0,EN,snowball,KEN,4,0
24217,92,Kenya,24/04/2020 11:06,24/04/2020 11:15,0.0,100.0,556.0,1.0,24/04/2020 11:15,100.0,...,1.0,6.0,5,0.0,0.0,EN,snowball,KEN,2,0
24218,92,Kenya,26/04/2020 10:16,26/04/2020 10:36,0.0,100.0,1218.0,1.0,26/04/2020 10:36,80.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0
24219,92,Kenya,23/04/2020 03:46,23/04/2020 04:13,0.0,100.0,1641.0,1.0,23/04/2020 04:13,3.0,...,1.0,3.0,4,NaN,0.0,EN,snowball,KEN,3,0


In [7]:
mask = (Covid51countries["countryname"] == "Spain") & (Covid51countries["language"] == "ES")
Covid51countries.loc[mask, "language"] = "ES-ES"

In [8]:
# We want to use on only one language of responses for each country
# Further, that language cannot be English

# Calculate the number of responses for each country
country_response_count = pd.DataFrame(Covid51countries[['countryname', 'ISO3']].value_counts())

#collect the languages which people used to respond from each country
languages_per_country = Covid51countries.groupby('countryname')['language'].value_counts().reset_index(name="count")

# mereg num resposes to languages people used
languages_per_country = languages_per_country.merge(
    country_response_count,
    on="countryname",
    how="left"
)

# Clean up
languages_per_country = languages_per_country.rename(columns={"count_x": "lang_responses", "count_y": "total_responses"})
languages_per_country["percent"] = languages_per_country["lang_responses"] / languages_per_country["total_responses"] * 100

languages_per_country

,countryname,language,lang_responses,total_responses,percent
0,Australia,ZH-S,181,378,47.883598
1,Australia,EN,156,378,41.269841
2,Australia,ID,12,378,3.174603
3,Australia,ZH-T,10,378,2.645503
4,Australia,VI,5,378,1.322751
...,...,...,...,...,...
495,Vietnam,ZH-S,2,338,0.591716
496,Vietnam,FR,1,338,0.295858
497,Vietnam,JA,1,338,0.295858
498,Vietnam,PL,1,338,0.295858


In [9]:
# get a table which translates some of the language codes into names
lang_names = pd.read_csv(Path("~/Projects/hypocognition/data/external/lang_names.csv"))
languages_per_country = languages_per_country.merge(lang_names, left_on="language", right_on="Code", how="left")
languages_per_country = languages_per_country.drop(columns=["Code"])
languages_per_country

,countryname,language,lang_responses,total_responses,percent,Language_Name
0,Australia,ZH-S,181,378,47.883598,Simplified Chinese
1,Australia,EN,156,378,41.269841,English
2,Australia,ID,12,378,3.174603,Indonesian
3,Australia,ZH-T,10,378,2.645503,Traditional Chinese
4,Australia,VI,5,378,1.322751,Vietnamese
...,...,...,...,...,...,...
495,Vietnam,ZH-S,2,338,0.591716,Simplified Chinese
496,Vietnam,FR,1,338,0.295858,French
497,Vietnam,JA,1,338,0.295858,Japanese
498,Vietnam,PL,1,338,0.295858,Polish


In [10]:
languages_per_country.to_csv(Path("~/Projects/hypocognition/data/processed/languages_per_country.csv").expanduser())

In [11]:
selection = pd.read_csv(Path("~/Projects/hypocognition/data/processed/sun_bila_selection.csv").expanduser())

In [12]:
# Lets filter the dataset using our new selection
filtered = Covid51countries.merge(
    selection,
    on=['countryname', 'language'],
    how='inner'
)
filtered

,country,countryname,StartDate,EndDate,Status,Progress,Duration,Finished,RecordedDate,home,...,datasource,ISO3,longstring,na_count,Unnamed: 0,lang_responses,total_responses,percent,Language_Name,Mapped_Target_Language
0,10,Australia,04/05/2020 19:34,04/05/2020 19:40,0.0,100.0,337.0,1.0,04/05/2020 19:40,8.0,...,snowball,AUS,3,0,0,181,378,47.883598,Simplified Chinese,Mandarin Chinese
1,10,Australia,04/05/2020 20:26,04/05/2020 20:35,0.0,100.0,530.0,1.0,04/05/2020 20:35,9.0,...,snowball,AUS,3,0,0,181,378,47.883598,Simplified Chinese,Mandarin Chinese
2,10,Australia,05/05/2020 21:54,05/05/2020 22:07,0.0,100.0,793.0,1.0,05/05/2020 22:07,29.0,...,snowball,AUS,5,0,0,181,378,47.883598,Simplified Chinese,Mandarin Chinese
3,10,Australia,04/05/2020 20:29,04/05/2020 20:34,0.0,100.0,285.0,1.0,04/05/2020 20:34,73.0,...,snowball,AUS,4,0,0,181,378,47.883598,Simplified Chinese,Mandarin Chinese
4,10,Australia,19/04/2020 20:21,19/04/2020 20:27,0.0,100.0,367.0,1.0,19/04/2020 20:27,100.0,...,snowball,AUS,3,0,0,181,378,47.883598,Simplified Chinese,Mandarin Chinese
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15515,91,Kazakhstan,25/04/2020 13:36,25/04/2020 13:44,0.0,100.0,470.0,1.0,25/04/2020 13:44,100.0,...,snowball,KAZ,3,0,28,279,291,95.876289,Russian,Russian
15516,91,Kazakhstan,06/05/2020 04:03,06/05/2020 04:18,0.0,100.0,884.0,1.0,06/05/2020 04:18,100.0,...,snowball,KAZ,3,0,28,279,291,95.876289,Russian,Russian
15517,91,Kazakhstan,06/05/2020 12:32,06/05/2020 12:41,0.0,100.0,573.0,1.0,06/05/2020 12:41,100.0,...,snowball,KAZ,4,0,28,279,291,95.876289,Russian,Russian
15518,91,Kazakhstan,06/05/2020 10:37,06/05/2020 10:49,0.0,100.0,701.0,1.0,06/05/2020 10:49,100.0,...,snowball,KAZ,5,0,28,279,291,95.876289,Russian,Russian


In [13]:
filtered = filtered[filtered["Mapped_Target_Language"]!="NONE"]

In [14]:
filtered[filtered['countryname']=="Spain"]['Language_Name'].unique()

array(['Spanish (Spain)'], dtype=object)

In [15]:
# Check to make sure it worked
filtered.groupby('countryname')['Language_Name'].unique()

countryname
Australia                                   [Simplified Chinese]
Brazil                                    [Brazilian Portuguese]
Bulgaria                                             [Bulgarian]
Canada                                      [Simplified Chinese]
Chile                                                  [Spanish]
China                                       [Simplified Chinese]
Colombia                                               [Spanish]
Croatia                                               [Croatian]
Curacao                                                  [Dutch]
Denmark                                                 [Danish]
Egypt                                                   [Arabic]
Finland                                                [Finnish]
France                                                  [French]
Georgia                                               [Georgian]
Germany                                                 [German]
Ghana        

In [16]:
filtered.groupby('countryname')['Mapped_Target_Language'].unique()

countryname
Australia                                 [Mandarin Chinese]
Brazil                                [Brazilian Portuguese]
Bulgaria                                         [Bulgarian]
Canada                                    [Mandarin Chinese]
Chile                                              [Spanish]
China                                     [Mandarin Chinese]
Colombia                                           [Spanish]
Croatia                           [Serbian-Croatian-Bosnian]
Curacao                                              [Dutch]
Denmark                                             [Danish]
Egypt                                               [Arabic]
Finland                                            [Finnish]
France                                              [French]
Georgia                                           [Georgian]
Germany                                             [German]
Ghana                                     [Mandarin Chinese]
Greece      

Goals
* For each distinct language, what is the lexical ellaboration for each of the 20 emotions?
* For each distinct language, is there any statistical significance in the distribution of the results of any particular emotion?

In [17]:
# drop the rest of the ~75 columns
filtered = filtered[['countryname', 'countryname', 'Language_Name', 'language', 'percent', 'admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']].copy()

In [18]:
emotions = ['admiration', 'calm', 'compassion',
       'determination', 'moved', 'gratitude', 'hope', 'love', 'relief',
       'pleasure', 'anger', 'anxiety', 'boredom', 'confusion', 'disgust',
       'fear', 'frustration', 'loneliness', 'regret', 'sadness']

In [19]:
# get the dataset of the dictionaries. We will look at how elaborated each of the twenty emotions are in each of the 39 languages
bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)
bila_nouns_full

/tmp/ipykernel_2912789/268126284.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bila_nouns_full = pd.read_csv(Path("~/Projects/hypocognition/data/raw/bila_long_noun_lemmatized_full.csv").expanduser(), index_col=0)


,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
4,chi.14718491,act,15.0,6.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,0.000000,0.000000,10499,0.000571,1.945910
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [20]:
with open(Path("~/Projects/hypocognition/data/external/stopwords.txt").expanduser()) as f:
    words = [line.strip() for line in f if line.strip()]
words

['word',
 'name',
 'form',
 'verb',
 'sound',
 'noun',
 'letter',
 'language',
 'class',
 'case',
 'mark',
 'comp',
 'note',
 'term',
 'meaning',
 'root',
 'sense',
 'prep',
 'speech',
 'subject',
 'sign',
 'character',
 'prop',
 'dictionary',
 'article',
 'adjective',
 'stem',
 'sentence',
 'particle',
 'dial',
 'person',
 'fig',
 'phrase',
 'section',
 'compound',
 'par',
 'con',
 'sub',
 'thing',
 'place',
 'part',
 'kind',
 'act',
 'way',
 'cause',
 'set',
 'side',
 'piece',
 'end',
 'state',
 'use',
 'round',
 'point',
 'manner',
 'object',
 'change',
 'matter',
 'action',
 'self',
 'top',
 'cover',
 'measure',
 'bit',
 'sort',
 'type',
 'good',
 'colour',
 'lot',
 'min',
 'middle',
 'degree',
 'member',
 'house',
 'home',
 'school',
 'work',
 'life',
 'mind',
 'number',
 'art',
 'world',
 'level',
 'style',
 'variety',
 'amount',
 'show',
 'specie',
 'man',
 'men',
 'woman',
 'women',
 'people',
 'family',
 'father',
 'mother',
 'son',
 'daughter',
 'child',
 'wife',
 'husband',


In [21]:
# remove stop words
bila_nouns_full = bila_nouns_full[~bila_nouns_full["word"].isin(words)]
bila_nouns_full

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,ability,2.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-1.424169,10499,0.000190,1.098612
1,chi.14718491,accomplice,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.156552,10499,0.000000,0.000000
2,chi.14718491,account,14.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-2.250524,10499,0.000857,2.302585
3,chi.14718491,acre,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.444142,10499,0.000000,0.000000
6,chi.14718491,actor,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-1.633890,10499,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5282195,wu.89119131142,prouds,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282196,wu.89119131142,facilitator,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282197,wu.89119131142,teacher-librarian,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000
5282198,wu.89119131142,pandani,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,0.000000,0.000000,33359,0.000000,0.000000


In [22]:
tot_words = bila_nouns_full.groupby('id')['word'].nunique().reset_index(name='Total_words_in_dict')
tot_words

,id,Total_words_in_dict
0,chi.14718491,8390
1,coo.31924067983704,8390
2,coo.31924099174934,8390
3,coo1.ark:/13960/t9m33d507,8390
4,dictionaria.daakaka_vonprince_daka1243,8390
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,8390
612,webonary.tampulma_asare_tamp1252,8390
613,webonary.turkmen_almamedow_turk1304,8390
614,wu.89017649658,8390


In [23]:
tot_counts = bila_nouns_full.groupby('id')['count'].sum().reset_index(name='Total_counts_in_dict')
tot_counts

,id,Total_counts_in_dict
0,chi.14718491,10499.0
1,coo.31924067983704,121674.0
2,coo.31924099174934,10358.0
3,coo1.ark:/13960/t9m33d507,20574.0
4,dictionaria.daakaka_vonprince_daka1243,1825.0
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,32836.0
612,webonary.tampulma_asare_tamp1252,4117.0
613,webonary.turkmen_almamedow_turk1304,27273.0
614,wu.89017649658,12018.0


In [24]:
# I think we will needs these later
dictionary_means = (
    bila_nouns_full
        .groupby('id', as_index=False)['count']
        .mean()
        .rename(columns={'count': 'dictionary_count_mean'})
)
dictionary_means

,id,dictionary_count_mean
0,chi.14718491,1.251371
1,coo.31924067983704,14.502265
2,coo.31924099174934,1.234565
3,coo1.ark:/13960/t9m33d507,2.452205
4,dictionaria.daakaka_vonprince_daka1243,0.217521
...,...,...
611,webonary.sursurunga_hutchisson_surs1246,3.913707
612,webonary.tampulma_asare_tamp1252,0.490703
613,webonary.turkmen_almamedow_turk1304,3.250656
614,wu.89017649658,1.432420


In [25]:
bila_nouns_full_emotions = bila_nouns_full[bila_nouns_full['word'].isin(emotions)].reset_index(drop=True)
bila_nouns_full_emotions

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,chi.14718491,fear,8.0,9.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-7.520776,-1.337584,10499,0.000857,2.302585
1,chi.14718491,pleasure,5.0,5.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.031819,-1.571145,10499,0.000476,1.791759
2,chi.14718491,regret,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-0.486657,10499,0.000190,1.098612
3,chi.14718491,admiration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-9.823849,-0.710507,10499,0.000000,0.000000
4,chi.14718491,anger,5.0,2.0,Nancowry,nanc1247,1884.0,A dictionary of the Nancowry dialect of the Ni...,"Printed at the Home Dept. Press, 1884.",0,Eurasia,Austroasiatic,"Austroasiatic, Nicobaric, Nuclear Nicobaric, C...",93.3921,7.94812,-8.725128,-2.335062,10499,0.000190,1.098612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11083,wu.89119131142,disgust,3.0,16.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-7.795743,3.869782,33359,0.000480,2.833213
11084,wu.89119131142,frustration,0.0,0.0,0,0,0.0,0,0,0,0,0,0,0.0000,0.00000,-10.629344,-0.596842,33359,0.000000,0.000000
11085,wu.89119131142,loneliness,3.0,4.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-9.019809,1.825273,33359,0.000120,1.609438
11086,wu.89119131142,relief,11.0,10.0,Herero,here1253,1989.0,An English-Herero dictionary : with an introdu...,"J.C. Juta, 1883.",0,Africa,Atlantic-Congo,"Atlantic-Congo, Volta-Congo, Benue-Congo, Bant...",20.5655,-21.02310,-8.231207,0.866433,33359,0.000300,2.397895


the dataset is missing two of the survey emotions, {'calm', 'moved'}

In [26]:
# already filtered out everything but the 20 emotions
# now we want to filter evrything but the 39 languages
# Where are lang_name and Language_Names the Same?
emotions_in_survey_languages_in_bila = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(filtered['Language_Name'].unique())].reset_index(drop=True)
emotions_in_survey_languages_in_bila

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
0,coo.31924067983704,fear,8.0,126.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-6.924086,1.952545,121674,0.001036,4.844187
1,coo.31924067983704,pleasure,5.0,107.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.086289,3.153327,121674,0.000879,4.682131
2,coo.31924067983704,regret,5.0,41.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-8.031261,2.670962,121674,0.000337,3.737670
3,coo.31924067983704,admiration,3.0,10.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.371276,-0.854986,121674,0.000082,2.397895
4,coo.31924067983704,anger,5.0,85.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.314243,0.579516,121674,0.000699,4.454347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,uva.x004877953,disgust,3.0,13.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-9.940215,-4.505457,282895,0.000046,2.639057
428,uva.x004877953,frustration,3.0,6.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.633387,-1.588076,282895,0.000021,1.945910
429,uva.x004877953,loneliness,3.0,7.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.499852,-1.873436,282895,0.000025,2.079442
430,uva.x004877953,relief,11.0,89.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-8.079201,3.907437,282895,0.000315,4.499810


In [27]:
# I asked gemini to map our study languages names to language names from the bila dataset
covid_to_bila_nouns_full_lang_name_mapping = pd.read_csv(Path("~/Projects/hypocognition/data/external/covid_to_full_bila_lang_name_mapping.csv"))
covid_to_bila_nouns_full_lang_name_mapping

,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,EN,English,NaN,"Midland American English, Singlish, Devon, Sus..."
2,ID,Indonesian,Indonesian,"Standard Malay, Central Malay, Baba Malay"
3,ZH-T,Traditional Chinese,Mandarin Chinese,"Yue Chinese, Min Dong Chinese, Min Nan Chinese..."
4,VI,Vietnamese,Vietnamese,Nung (Viet Nam)
5,DA,Danish,Danish,NaN
6,AR,Arabic,Arabic,"Egyptian Arabic, Levantine Arabic, Moroccan Ar..."
7,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
8,AFRI,Afrikaans,Afrikaans,Dutch
9,ES-ES,Spanish (Spain),Spanish,NaN


In [28]:
# Update selection table 
selection_in_bila =selection.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on="study_language_names",
    how="left"
)

selection_in_bila

,Unnamed: 0,countryname,language,lang_responses,total_responses,percent,Language_Name,Mapped_Target_Language,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,0,Australia,ZH-S,181,378,47.883598,Simplified Chinese,Mandarin Chinese,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,1,Brazil,PT-BR,306,325,94.153846,Brazilian Portuguese,Brazilian Portuguese,PT-BR,Brazilian Portuguese,Brazilian Portuguese,Portuguese
2,2,Bulgaria,BG,269,276,97.463768,Bulgarian,Bulgarian,BG,Bulgarian,Bulgarian,NaN
3,3,Canada,ZH-S,26,286,9.090909,Simplified Chinese,Mandarin Chinese,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
4,4,Chile,ES,469,493,95.131846,Spanish,Spanish,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
5,5,China,ZH-S,1988,2009,98.954704,Simplified Chinese,Mandarin Chinese,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
6,6,Colombia,ES,122,237,51.476793,Spanish,Spanish,ES,Spanish,Spanish,"Latin American Spanish, Mexican Spanish"
7,7,Croatia,HR,524,528,99.242424,Croatian,Serbian-Croatian-Bosnian,HR,Croatian,Serbian-Croatian-Bosnian,NaN
8,8,Curacao,NL,180,235,76.595745,Dutch,Dutch,NL,Dutch,Dutch,Western Flemish
9,9,Denmark,DA,148,233,63.519313,Danish,Danish,DA,Danish,Danish,NaN


In [29]:
selection_in_bila.columns

Index(['Unnamed: 0', 'countryname', 'language', 'lang_responses',
       'total_responses', 'percent', 'Language_Name', 'Mapped_Target_Language',
       'code', 'study_language_names', 'bila_language_name_mapping',
       'possible_alternative_bila_language_name_mappings'],
      dtype='object')

In [30]:
selection_in_bila = selection_in_bila[['countryname', 'language','study_language_names', 'bila_language_name_mapping', 'lang_responses', 'total_responses',
       'percent',
       'possible_alternative_bila_language_name_mappings']]

In [31]:
selection_in_bila.to_csv("~/Projects/hypocognition/data/processed/MAX_selection_in_bila.csv")

In [32]:
# Add a language name map for the the bila dataset language names
filtered_mapped =filtered.merge(
    covid_to_bila_nouns_full_lang_name_mapping,
    left_on="Language_Name",
    right_on="study_language_names",
    how="left"
)

filtered_mapped

,countryname,countryname,Language_Name,language,percent,admiration,calm,compassion,determination,moved,...,disgust,fear,frustration,loneliness,regret,sadness,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,Australia,Simplified Chinese,ZH-S,47.883598,5.0,3.0,3.0,5.0,4.0,...,2.0,1.0,2.0,1.0,1.0,1.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Australia,Australia,Simplified Chinese,ZH-S,47.883598,2.0,3.0,3.0,0.0,4.0,...,3.0,3.0,1.0,1.0,2.0,2.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
2,Australia,Australia,Simplified Chinese,ZH-S,47.883598,3.0,3.0,0.0,0.0,0.0,...,2.0,0.0,0.0,0.0,0.0,0.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
3,Australia,Australia,Simplified Chinese,ZH-S,47.883598,6.0,1.0,4.0,4.0,4.0,...,4.0,4.0,3.0,6.0,1.0,2.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
4,Australia,Australia,Simplified Chinese,ZH-S,47.883598,0.0,6.0,6.0,6.0,2.0,...,2.0,4.0,2.0,1.0,0.0,3.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15231,Kazakhstan,Kazakhstan,Russian,RU,95.876289,1.0,3.0,3.0,1.0,5.0,...,2.0,3.0,2.0,3.0,3.0,2.0,RU,Russian,Russian,Belarusian
15232,Kazakhstan,Kazakhstan,Russian,RU,95.876289,3.0,3.0,6.0,2.0,6.0,...,1.0,5.0,2.0,2.0,6.0,5.0,RU,Russian,Russian,Belarusian
15233,Kazakhstan,Kazakhstan,Russian,RU,95.876289,3.0,3.0,3.0,3.0,4.0,...,3.0,2.0,5.0,4.0,4.0,3.0,RU,Russian,Russian,Belarusian
15234,Kazakhstan,Kazakhstan,Russian,RU,95.876289,2.0,3.0,6.0,2.0,2.0,...,3.0,3.0,3.0,1.0,6.0,4.0,RU,Russian,Russian,Belarusian


In [33]:
filtered_mapped = filtered_mapped[filtered_mapped['bila_language_name_mapping'].notna()]
filtered_mapped

,countryname,countryname,Language_Name,language,percent,admiration,calm,compassion,determination,moved,...,disgust,fear,frustration,loneliness,regret,sadness,code,study_language_names,bila_language_name_mapping,possible_alternative_bila_language_name_mappings
0,Australia,Australia,Simplified Chinese,ZH-S,47.883598,5.0,3.0,3.0,5.0,4.0,...,2.0,1.0,2.0,1.0,1.0,1.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
1,Australia,Australia,Simplified Chinese,ZH-S,47.883598,2.0,3.0,3.0,0.0,4.0,...,3.0,3.0,1.0,1.0,2.0,2.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
2,Australia,Australia,Simplified Chinese,ZH-S,47.883598,3.0,3.0,0.0,0.0,0.0,...,2.0,0.0,0.0,0.0,0.0,0.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
3,Australia,Australia,Simplified Chinese,ZH-S,47.883598,6.0,1.0,4.0,4.0,4.0,...,4.0,4.0,3.0,6.0,1.0,2.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
4,Australia,Australia,Simplified Chinese,ZH-S,47.883598,0.0,6.0,6.0,6.0,2.0,...,2.0,4.0,2.0,1.0,0.0,3.0,ZH-S,Simplified Chinese,Mandarin Chinese,"Beijing Mandarin, Wu Chinese"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15231,Kazakhstan,Kazakhstan,Russian,RU,95.876289,1.0,3.0,3.0,1.0,5.0,...,2.0,3.0,2.0,3.0,3.0,2.0,RU,Russian,Russian,Belarusian
15232,Kazakhstan,Kazakhstan,Russian,RU,95.876289,3.0,3.0,6.0,2.0,6.0,...,1.0,5.0,2.0,2.0,6.0,5.0,RU,Russian,Russian,Belarusian
15233,Kazakhstan,Kazakhstan,Russian,RU,95.876289,3.0,3.0,3.0,3.0,4.0,...,3.0,2.0,5.0,4.0,4.0,3.0,RU,Russian,Russian,Belarusian
15234,Kazakhstan,Kazakhstan,Russian,RU,95.876289,2.0,3.0,6.0,2.0,2.0,...,3.0,3.0,3.0,1.0,6.0,4.0,RU,Russian,Russian,Belarusian


In [34]:
filtered_mapped["study_language_names"].unique()

array(['Simplified Chinese', 'Malay', 'Russian', 'Dutch', 'French',
       'Spanish (Spain)', 'Serbian', 'Afrikaans', 'Swedish', 'Arabic',
       'Traditional Chinese (Taiwan)', 'Spanish', 'Turkish', 'Ukrainian',
       'Vietnamese', 'Brazilian Portuguese', 'Bulgarian', 'Croatian',
       'Danish', 'Finnish', 'Georgian', 'German', 'Greek',
       'Traditional Chinese', 'Hungarian', 'Icelandic', 'Marathi',
       'Indonesian', 'Persian', 'Hebrew', 'Italian', 'Japanese'],
      dtype=object)

In [35]:
filtered_mapped["study_language_names"].nunique()

32

In [36]:
filtered_mapped["bila_language_name_mapping"].nunique()

28

In [37]:
filtered_mapped['countryname'].nunique()

countryname    49
countryname    49
dtype: int64

In [38]:
filtered_mapped = filtered_mapped.loc[:, ~filtered_mapped.columns.duplicated()]

In [39]:
bila_nouns_full_emotions_filtered = bila_nouns_full_emotions[bila_nouns_full_emotions['langname'].isin(covid_to_bila_nouns_full_lang_name_mapping['bila_language_name_mapping'].values)]
bila_nouns_full_emotions_filtered

,id,word,nsenses,count,langname,glottocode,year,title,imprint,author,area,langfamily,affiliation,longitude,latitude,estimate,regression_elaboration,dictsize_data,simple_elaboration,log_count
18,coo.31924067983704,fear,8.0,126.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-6.924086,1.952545,121674,0.001036,4.844187
19,coo.31924067983704,pleasure,5.0,107.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.086289,3.153327,121674,0.000879,4.682131
20,coo.31924067983704,regret,5.0,41.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-8.031261,2.670962,121674,0.000337,3.737670
21,coo.31924067983704,admiration,3.0,10.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-9.371276,-0.854986,121674,0.000082,2.397895
22,coo.31924067983704,anger,5.0,85.0,Turkish,nucl1301,1991.0,Redhouse yeni Türkçe-İngilizce sözlük = New Re...,"Redhouse Yayınevi, 1991, c1968.",0,Eurasia,Turkic,"Turkic, Common Turkic, Oghuz, Nuclear Oghuz, W...",32.8667,39.8667,-7.314243,0.579516,121674,0.000699,4.454347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10417,uva.x004877953,disgust,3.0,13.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-9.940215,-4.505457,282895,0.000046,2.639057
10418,uva.x004877953,frustration,3.0,6.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.633387,-1.588076,282895,0.000021,1.945910
10419,uva.x004877953,loneliness,3.0,7.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-10.499852,-1.873436,282895,0.000025,2.079442
10420,uva.x004877953,relief,11.0,89.0,Japanese,nucl1643,1974.0,Kenkyusha's new Japanese-English dictionary. K...,Kenkyusha [1974],0,Eurasia,Japonic,"Japonic, Japanesic, Japan-Taiwan Japanese",135.0000,35.0000,-8.079201,3.907437,282895,0.000315,4.499810


In [40]:
bila_nouns_full_emotions_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 18 to 10421
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      713 non-null    object 
 1   word                    713 non-null    object 
 2   nsenses                 713 non-null    float64
 3   count                   713 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    float64
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    floa

In [41]:
# add the rows of the emotions words which don't appear in each dictionary
bila_emotions = bila_nouns_full_emotions_filtered['word'].unique()
full_index = pd.MultiIndex.from_product(
    [bila_nouns_full_emotions_filtered["id"].unique(), bila_emotions],
    names=["id", "word"]
)


In [42]:

bila_nouns_full_emotions_filtered_full = (
    bila_nouns_full_emotions_filtered
    .set_index(["id", "word"])
    .reindex(full_index)
    .reset_index()
)
num_cols = ["nsenses", "count", "log_count"]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 713 non-null    float64
 3   count                   713 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    float64
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    fl

In [43]:

#set the number columns to 0 where NaN
bila_nouns_full_emotions_filtered_full[num_cols] = bila_nouns_full_emotions_filtered_full[num_cols].fillna(0)
meta_cols = [
    "langname", "glottocode", "year", "title", "imprint", "author",
    "area", "langfamily", "affiliation", "longitude", "latitude"
]
bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 738 non-null    float64
 3   count                   738 non-null    float64
 4   langname                713 non-null    object 
 5   glottocode              713 non-null    object 
 6   year                    713 non-null    float64
 7   title                   713 non-null    object 
 8   imprint                 713 non-null    object 
 9   author                  713 non-null    object 
 10  area                    713 non-null    object 
 11  langfamily              713 non-null    object 
 12  affiliation             713 non-null    object 
 13  longitude               713 non-null    float64
 14  latitude                713 non-null    fl

In [44]:
# fill the those zero rows with dictionary meta -data
bila_nouns_full_emotions_filtered_full[meta_cols] = (
    bila_nouns_full_emotions_filtered_full
    .groupby("id")[meta_cols]
    .transform("first")
)

bila_nouns_full_emotions_filtered_full.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   id                      738 non-null    object 
 1   word                    738 non-null    object 
 2   nsenses                 738 non-null    float64
 3   count                   738 non-null    float64
 4   langname                738 non-null    object 
 5   glottocode              738 non-null    object 
 6   year                    738 non-null    float64
 7   title                   738 non-null    object 
 8   imprint                 738 non-null    object 
 9   author                  738 non-null    object 
 10  area                    738 non-null    object 
 11  langfamily              738 non-null    object 
 12  affiliation             738 non-null    object 
 13  longitude               738 non-null    float64
 14  latitude                738 non-null    fl

In [45]:
elab_per_emotion = bila_nouns_full_emotions_filtered_full[['langname',   'glottocode','word', 'count',  'id', 'year',"simple_elaboration", "regression_elaboration",	"dictsize_data"	]]
elab_per_emotion

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0
...,...,...,...,...,...,...,...,...,...
733,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0
734,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0
735,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0
736,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0


In [46]:
elab_per_emotion.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 738 entries, 0 to 737
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   langname                738 non-null    object 
 1   glottocode              738 non-null    object 
 2   word                    738 non-null    object 
 3   count                   738 non-null    float64
 4   id                      738 non-null    object 
 5   year                    738 non-null    float64
 6   simple_elaboration      713 non-null    float64
 7   regression_elaboration  713 non-null    float64
 8   dictsize_data           713 non-null    float64
dtypes: float64(5), object(4)
memory usage: 52.0+ KB


In [47]:
stats_by_country = (
    filtered_mapped
    .groupby(['countryname', 'bila_language_name_mapping'])[emotions]
    .agg(['mean', 'std'])
)
stats_by_country = pd.DataFrame(stats_by_country)
stats_by_country

admiration  \
                                                                mean   
countryname                    bila_language_name_mapping              
Australia                      Mandarin Chinese             3.413408   
Brazil                         Brazilian Portuguese         3.540984   
Bulgaria                       Bulgarian                    2.498141   
Canada                         Mandarin Chinese             3.076923   
Chile                          Spanish                      2.867521   
China                          Mandarin Chinese             4.365940   
Colombia                       Spanish                      3.409836   
Croatia                        Serbian-Croatian-Bosnian     2.552581   
Curacao                        Dutch                        3.627778   
Denmark                        Danish                       2.668919   
Egypt                          Arabic                       2.547472   
Finland                        Finnish                      2.738462   
France                         French                       3.357977   
Georgia                        Georgian                     2.523605   
Germany                        German                       2.804124   
Ghana                          Mandarin Chinese             6.000000   
Greece                         Greek                        2.328889   
Hong Kong (S.A.R)              Mandarin Chinese             2.669065   
Hungary                        Hungarian                    2.679688   
Iceland                        Icelandic                    3.469741   
India                          Marathi                      3.500000   
Indonesia                      Indonesian                   3.201863   
Iran                           Persian                      2.347826   
Ireland                        Serbian-Croatian-Bosnian     0.666667   
Israel                         Hebrew                       1.569536   
Italy                          Italian                      2.966981   
Japan                          Japanese                     2.602176   
Jordan                         Arabic                       3.108225   
Kazakhstan                     Russian                      2.438849   
Malaysia                       Standard Malay               3.793548   
Mongolia                       Russian                      2.461538   
Netherlands                    Dutch                        3.325048   
New Zealand                    Mandarin Chinese             3.215686   
Pakistan                       French                       2.000000   
Peru                           Spanish                      3.706161   
Russia                         Russian                      2.553043   
Serbia                         Serbian-Croatian-Bosnian     2.712644   
Singapore                      Mandarin Chinese             3.250000   
South Africa                   Afrikaans                    2.946429   
Spain                          Spanish                      4.021390   
Sweden                         Swedish                      3.300000   
Syria                          Arabic                       1.849372   
Taiwan                         Mandarin Chinese             3.993307   
Trinidad and Tobago            Spanish                      3.000000   
Turkey                         Turkish                      2.027888   
Ukraine                        Ukrainian                    2.856489   
United Kingdom (UK)            Mandarin Chinese             4.088235   
United States of America (USA) Mandarin Chinese             3.044444   
Vietnam                        Vietnamese                   4.219178   

                                                                         calm  \
                                                                std      mean   
countryname                    bila_language_name_mapping                       
Australia                      Mandarin Chinese            1.892338  3

In [48]:
stats_by_country.columns = [
    f"{emotion}_{stat}" for emotion, stat in stats_by_country.columns
]
stats_by_country = stats_by_country.reset_index()
stats_by_country

,countryname,bila_language_name_mapping,admiration_mean,admiration_std,calm_mean,calm_std,compassion_mean,compassion_std,determination_mean,determination_std,...,fear_mean,fear_std,frustration_mean,frustration_std,loneliness_mean,loneliness_std,regret_mean,regret_std,sadness_mean,sadness_std
0,Australia,Mandarin Chinese,3.413408,1.892338,3.849162,1.609186,4.055556,1.780472,3.887640,1.604642,...,1.878453,1.631846,1.972067,1.730202,1.900000,1.818895,1.309392,1.473232,1.955556,1.680723
1,Brazil,Brazilian Portuguese,3.540984,1.745047,3.421569,1.502585,4.633987,1.319471,3.725490,1.593986,...,3.603279,1.757500,3.437908,1.941201,2.518033,2.069635,1.934426,1.881951,3.321311,1.833991
2,Bulgaria,Bulgarian,2.498141,1.980548,3.334572,1.712431,4.313433,1.590788,3.565056,1.684113,...,2.561338,1.954938,3.029740,2.089207,2.334572,2.154385,2.929104,2.009018,3.252788,1.930519
3,Canada,Mandarin Chinese,3.076923,2.018377,3.576923,1.447013,4.153846,1.689788,3.615385,1.328967,...,2.500000,1.881489,2.769231,1.795721,2.807692,1.720912,1.884615,1.451259,3.192308,1.720912
4,Chile,Spanish,2.867521,1.901080,2.899358,1.546330,3.893390,1.693798,3.370450,1.700109,...,3.528785,1.836746,4.119914,1.731608,2.869658,2.067930,2.352564,1.969156,3.799145,1.745670
5,China,Mandarin Chinese,4.365940,1.736698,3.914807,1.683731,4.240976,1.701800,4.165992,1.693406,...,1.809500,1.726083,1.815642,1.763461,1.812437,1.826492,1.477009,1.701083,2.256709,1.848471
6,Colombia,Spanish,3.409836,1.974022,3.418033,1.419115,4.024590,1.678563,3.818182,1.653280,...,3.024590,1.943258,3.409836,1.897167,2.578512,2.170837,1.721311,1.601973,3.459016,1.916210
7,Croatia,Serbian-Croatian-Bosnian,2.552581,1.680006,3.332061,1.485572,2.904398,1.650043,3.508604,1.438863,...,2.739962,1.825025,3.430210,1.808983,2.692748,1.952364,2.795802,1.713266,2.990458,1.759406
8,Curacao,Dutch,3.627778,1.743401,3.961111,1.607816,4.777778,1.174951,4.000000,1.549914,...,2.477778,1.933115,3.005556,1.877527,1.894444,1.853570,1.350000,1.645970,2.622222,1.906049
9,Denmark,Danish,2.668919,1.513605,3.445946,1.531012,4.067568,1.450602,3.351351,1.483996,...,1.824324,1.610832,3.148649,1.852980,2.277027,2.026427,1.479730,1.655622,1.601351,1.732993


In [49]:
stats_long = (
    stats_by_country
    .set_index(['countryname', 'bila_language_name_mapping'])
    .filter(regex='_(mean|std)$')
    .stack()
    .reset_index()
)

stats_long[['word', 'stat']] = stats_long['level_2'].str.rsplit('_', n=1, expand=True)
stats_long = stats_long.rename(columns={0: 'value'}).drop(columns='level_2')

stats_long = (
    stats_long
    .pivot_table(
        index=['countryname', 'bila_language_name_mapping', 'word'],
        columns='stat',
        values='value'
    )
    .reset_index()
)


In [50]:

merged = elab_per_emotion.merge(
    stats_long,
    left_on=['langname', 'word'],
    right_on=['bila_language_name_mapping', 'word'],
    how='left'
)

merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0,Turkey,Turkish,2.948617,1.892269
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0,Turkey,Turkish,3.083665,1.643463
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0,Turkey,Turkish,1.768000,1.788567
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1111,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0,Japan,Japanese,2.854701,1.747655
1112,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0,Japan,Japanese,3.111888,1.729330
1113,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0,Japan,Japanese,2.007764,1.817621
1114,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428


In [51]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1116 entries, 0 to 1115
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    1116 non-null   object 
 1   glottocode                  1116 non-null   object 
 2   word                        1116 non-null   object 
 3   count                       1116 non-null   float64
 4   id                          1116 non-null   object 
 5   year                        1116 non-null   float64
 6   simple_elaboration          1091 non-null   float64
 7   regression_elaboration      1091 non-null   float64
 8   dictsize_data               1091 non-null   float64
 9   countryname                 882 non-null    object 
 10  bila_language_name_mapping  882 non-null    object 
 11  mean                        882 non-null    float64
 12  std                         845 non-null    float64
dtypes: float64(7), object(6)
memory u

In [52]:
# add the dictionary_count_mean column
merged = merged.merge(
    dictionary_means,
    left_on='id',
    right_on='id',
    how='left'
)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,mean,std,dictionary_count_mean
0,Turkish,nucl1301,fear,126.0,coo.31924067983704,1991.0,0.001036,1.952545,121674.0,Turkey,Turkish,2.948617,1.892269,14.502265
1,Turkish,nucl1301,pleasure,107.0,coo.31924067983704,1991.0,0.000879,3.153327,121674.0,Turkey,Turkish,3.083665,1.643463,14.502265
2,Turkish,nucl1301,regret,41.0,coo.31924067983704,1991.0,0.000337,2.670962,121674.0,Turkey,Turkish,1.768000,1.788567,14.502265
3,Turkish,nucl1301,admiration,10.0,coo.31924067983704,1991.0,0.000082,-0.854986,121674.0,Turkey,Turkish,2.027888,1.760460,14.502265
4,Turkish,nucl1301,anger,85.0,coo.31924067983704,1991.0,0.000699,0.579516,121674.0,Turkey,Turkish,3.460000,1.871473,14.502265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1111,Japanese,nucl1643,disgust,13.0,uva.x004877953,1974.0,0.000046,-4.505457,282895.0,Japan,Japanese,2.854701,1.747655,33.718117
1112,Japanese,nucl1643,frustration,6.0,uva.x004877953,1974.0,0.000021,-1.588076,282895.0,Japan,Japanese,3.111888,1.729330,33.718117
1113,Japanese,nucl1643,loneliness,7.0,uva.x004877953,1974.0,0.000025,-1.873436,282895.0,Japan,Japanese,2.007764,1.817621,33.718117
1114,Japanese,nucl1643,relief,89.0,uva.x004877953,1974.0,0.000315,3.907437,282895.0,Japan,Japanese,2.403263,1.417428,33.718117


In [53]:
merged = merged[merged['countryname'].notna()]

In [54]:
merged.columns

Index(['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'mean', 'std',
       'dictionary_count_mean'],
      dtype='object')

In [55]:
# clarify the meaning of mean
merged = merged.rename(columns={'mean': "response_mean"})
# Clarify what std we're talking about
merged = merged.rename(columns={'std': "response_std"})
# drop a bunch of columns

merged = merged[['langname', 'glottocode', 'word', 'count', 'id', 'year',
       'simple_elaboration', 'regression_elaboration', 'dictsize_data',
       'countryname', 'bila_language_name_mapping', 'response_mean',
       'response_std', 'dictionary_count_mean' ]]
merged = merged.sort_values(by=["langname", 'word'])
merged


,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
417,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
418,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
421,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
431,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
419,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
595,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
596,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
610,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [56]:
merged = merged.reset_index(drop=True)
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
878,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
879,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
880,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [57]:
merged[merged['countryname']=="China"]

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
380,Mandarin Chinese,mand1415,admiration,24.0,uc1.l0088545033,2004.0,0.000236,3.652824,101618.0,China,Mandarin Chinese,4.365940,1.736698,12.1118
390,Mandarin Chinese,mand1415,anger,59.0,uc1.l0088545033,2004.0,0.000581,-0.995820,101618.0,China,Mandarin Chinese,1.956434,1.812030,12.1118
400,Mandarin Chinese,mand1415,anxiety,33.0,uc1.l0088545033,2004.0,0.000325,1.855676,101618.0,China,Mandarin Chinese,2.632892,1.879226,12.1118
410,Mandarin Chinese,mand1415,boredom,5.0,uc1.l0088545033,2004.0,0.000049,0.545344,101618.0,China,Mandarin Chinese,2.329782,1.948270,12.1118
420,Mandarin Chinese,mand1415,compassion,9.0,uc1.l0088545033,2004.0,0.000089,-1.569685,101618.0,China,Mandarin Chinese,4.240976,1.701800,12.1118
430,Mandarin Chinese,mand1415,confusion,19.0,uc1.l0088545033,2004.0,0.000187,-2.766752,101618.0,China,Mandarin Chinese,2.270421,1.809292,12.1118
440,Mandarin Chinese,mand1415,determination,8.0,uc1.l0088545033,2004.0,0.000079,-1.131749,101618.0,China,Mandarin Chinese,4.165992,1.693406,12.1118
450,Mandarin Chinese,mand1415,disgust,3.0,uc1.l0088545033,2004.0,0.000030,-2.957027,101618.0,China,Mandarin Chinese,1.745197,1.751591,12.1118
460,Mandarin Chinese,mand1415,fear,89.0,uc1.l0088545033,2004.0,0.000876,-0.020181,101618.0,China,Mandarin Chinese,1.809500,1.726083,12.1118
470,Mandarin Chinese,mand1415,frustration,7.0,uc1.l0088545033,2004.0,0.000069,1.444598,101618.0,China,Mandarin Chinese,1.815642,1.763461,12.1118


In [58]:
merged['word'].nunique()

18

In [59]:
merged = merged.sort_values(by=["langname", 'countryname','word'])
merged = merged.reset_index(drop=True)
merged = merged.fillna(0)


In [60]:
merged

,langname,glottocode,word,count,id,year,simple_elaboration,regression_elaboration,dictsize_data,countryname,bila_language_name_mapping,response_mean,response_std,dictionary_count_mean
0,Afrikaans,afri1274,admiration,6.0,mdp.39015054154474,1999.0,0.000028,-3.291632,212992.0,South Africa,Afrikaans,2.946429,1.712904,25.386412
1,Afrikaans,afri1274,anger,55.0,mdp.39015054154474,1999.0,0.000258,-6.741147,212992.0,South Africa,Afrikaans,2.785714,1.803043,25.386412
2,Afrikaans,afri1274,anxiety,30.0,mdp.39015054154474,1999.0,0.000141,-2.655449,212992.0,South Africa,Afrikaans,3.535714,1.765171,25.386412
3,Afrikaans,afri1274,boredom,5.0,mdp.39015054154474,1999.0,0.000023,-1.176312,212992.0,South Africa,Afrikaans,2.241071,1.889971,25.386412
4,Afrikaans,afri1274,compassion,15.0,mdp.39015054154474,1999.0,0.000070,-2.918472,212992.0,South Africa,Afrikaans,4.196429,1.505780,25.386412
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
877,Vietnamese,viet1252,love,308.0,uc1.31210024487306,2003.0,0.001143,-1.138291,269366.0,Vietnam,Vietnamese,4.786942,1.311475,32.105602
878,Vietnamese,viet1252,pleasure,216.0,uc1.31210024487306,2003.0,0.000802,3.511485,269366.0,Vietnam,Vietnamese,2.832192,1.885719,32.105602
879,Vietnamese,viet1252,regret,59.0,uc1.31210024487306,2003.0,0.000219,0.052866,269366.0,Vietnam,Vietnamese,1.534247,1.709743,32.105602
880,Vietnamese,viet1252,relief,74.0,uc1.31210024487306,2003.0,0.000275,2.404758,269366.0,Vietnam,Vietnamese,3.199313,1.720505,32.105602


In [61]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882 entries, 0 to 881
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   langname                    882 non-null    object 
 1   glottocode                  882 non-null    object 
 2   word                        882 non-null    object 
 3   count                       882 non-null    float64
 4   id                          882 non-null    object 
 5   year                        882 non-null    float64
 6   simple_elaboration          882 non-null    float64
 7   regression_elaboration      882 non-null    float64
 8   dictsize_data               882 non-null    float64
 9   countryname                 882 non-null    object 
 10  bila_language_name_mapping  882 non-null    object 
 11  response_mean               882 non-null    float64
 12  response_std                882 non-null    float64
 13  dictionary_count_mean       882 non

In [62]:
merged.to_csv(Path("~/Projects/hypocognition/data/processed/MAX_covid_bila_merge.csv").expanduser())

"Moving forward, could you create a table with the following columns (broken down into the 36 samples):
* The sample country
* The language
* The correlation between the log count and the response means
* The correlation between the log count and the response SDs
* The correlation between the log count and the absolute distance of the response means from the scale midpoint

In [63]:
merged['response_mean'].mean()

np.float64(3.027895463883044)

In [64]:

# log count (add 1 to avoid log(0) just in case)
merged2 = merged.copy()
merged2["log_count"] = np.log(merged2["count"] + 1)

def corr_within_group(df, x, y):
    return df[x].corr(df[y])

countries = merged2['countryname'].unique().tolist()

for c in countries:
    df = merged2[merged2['countryname']==c]
    print(c,df['log_count'].corr(df['response_mean']), df['log_count'].corr(df['response_std']))





South Africa 0.09148865168961892 -0.12653514046747824
Egypt -0.11433067618319566 0.13329119935570388
Jordan -0.22730182023960746 0.22199851220158942
Syria 0.08928912107744537 -0.29449735181619413
Brazil 0.1624355100367705 -0.09586451876984464
Bulgaria 0.2914576122549996 -0.510940348557617
Denmark -0.3228657980051242 -0.19173518683367818
Curacao -0.22717185923377653 -0.01279205981994988
Netherlands -0.22757861410314467 -0.30170812809299574
Finland 0.0720360849933431 -0.29966805278456204
France -0.12485678545510513 0.16444273794734743
Pakistan 0.15151736280403277 0.16453879766845736
Georgia -0.09763318852656655 0.08255505707391594
Germany 0.17814793850857474 -0.09156428792952183
Greece 0.155470088914526 -0.2718482113675404
Israel -0.19576632870638763 0.22289222905270406
Hungary 0.27087397624337056 -0.3665163766184561
Iceland 0.13037213359602945 -0.3824791026602647
Indonesia -0.01639818458273091 -0.10834449797267735
Italy -0.02900174620842426 -0.3471079511701444
Japan 0.0804512593809085 -

/data/home/asher.katz/miniconda3/envs/hypenv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/data/home/asher.katz/miniconda3/envs/hypenv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [65]:
import numpy as np

merged2 = merged.copy()

# log count
merged2["log_count"] = np.log(merged2["count"] + 1)

# absolute distance from midpoint
SCALE_MIDPOINT = merged2["response_mean"].mean()
merged2["abs_dist_midpoint"] = (
    merged2["response_mean"] - SCALE_MIDPOINT
).abs()

result = (
    merged2
    .groupby(["countryname", "langname", "id"])
    .agg(
        corr_logcount_response_mean=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_mean"]
            )
        ),
        corr_logcount_response_sd=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "response_std"]
            )
        ),
        corr_logcount_abs_dist_midpoint=(
            "log_count",
            lambda x: x.corr(
                merged2.loc[x.index, "abs_dist_midpoint"]
            )
        ),

    )
    .reset_index()
)


/data/home/asher.katz/miniconda3/envs/hypenv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/data/home/asher.katz/miniconda3/envs/hypenv/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [66]:
result

,countryname,langname,id,corr_logcount_response_mean,corr_logcount_response_sd,corr_logcount_abs_dist_midpoint
0,Australia,Mandarin Chinese,uc1.l0088545033,0.164092,-0.050474,-0.050495
1,Brazil,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.060763
2,Bulgaria,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.256754
3,Canada,Mandarin Chinese,uc1.l0088545033,-0.053859,0.048283,0.283331
4,Chile,Spanish,umn.31951d01452389x,0.199989,-0.153504,0.284414
5,China,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.107092
6,Colombia,Spanish,umn.31951d01452389x,0.029407,-0.270589,-0.066229
7,Croatia,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.363570,-0.037434,-0.237194
8,Curacao,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.209157
9,Denmark,Danish,uc1.b3832576,-0.322866,-0.191735,0.325635


In [117]:
result.to_csv(Path("~/Projects/hypocognition/data/processed/MAX_corr_covid_bila.csv").expanduser())

In [69]:
corr1 = pd.read_csv(Path("~/Projects/hypocognition/data/processed/covid_bila_correlation (7).csv").expanduser(), index_col=0)
corr1

,countryname,langname,id,corr_logcount_response_mean,corr_logcount_response_sd,corr_logcount_abs_dist_midpoint
0,Brazil,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.058824
1,Bulgaria,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.265190
2,Chile,Latin American Spanish,mdp.39015050181174,0.371436,-0.362343,0.241720
3,China,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.116186
4,Colombia,Latin American Spanish,mdp.39015050181174,0.236561,-0.175261,-0.137527
5,Croatia,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.363570,-0.037434,-0.247292
6,Curacao,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.211604
7,Denmark,Danish,uc1.b3832576,-0.322866,-0.191735,0.313704
8,Egypt,Arabic,mdp.39015043036436,-0.114331,0.133291,-0.338161
9,Finland,Finnish,mdp.39015059174667,0.072036,-0.299668,-0.086969


In [70]:
both = result.merge(
    corr1,
    on="countryname"
)
both

,countryname,langname_x,id_x,corr_logcount_response_mean_x,corr_logcount_response_sd_x,corr_logcount_abs_dist_midpoint_x,langname_y,id_y,corr_logcount_response_mean_y,corr_logcount_response_sd_y,corr_logcount_abs_dist_midpoint_y
0,Brazil,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.060763,Brazilian Portuguese,txu.059173018640743,0.162436,-0.095865,0.058824
1,Bulgaria,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.256754,Bulgarian,mdp.39015058560494,0.291458,-0.510940,0.265190
2,Chile,Spanish,umn.31951d01452389x,0.199989,-0.153504,0.284414,Latin American Spanish,mdp.39015050181174,0.371436,-0.362343,0.241720
3,China,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.107092,Mandarin Chinese,uc1.l0088545033,0.191494,-0.404744,0.116186
4,Colombia,Spanish,umn.31951d01452389x,0.029407,-0.270589,-0.066229,Latin American Spanish,mdp.39015050181174,0.236561,-0.175261,-0.137527
5,Croatia,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.363570,-0.037434,-0.237194,Serbian-Croatian-Bosnian,mdp.39015012891506,-0.363570,-0.037434,-0.247292
6,Curacao,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.209157,Dutch,uc1.31822004083036,-0.227172,-0.012792,-0.211604
7,Denmark,Danish,uc1.b3832576,-0.322866,-0.191735,0.325635,Danish,uc1.b3832576,-0.322866,-0.191735,0.313704
8,Egypt,Arabic,mdp.39015043036436,-0.114331,0.133291,-0.347282,Arabic,mdp.39015043036436,-0.114331,0.133291,-0.338161
9,Finland,Finnish,mdp.39015059174667,0.072036,-0.299668,-0.090822,Finnish,mdp.39015059174667,0.072036,-0.299668,-0.086969


In [71]:
both.columns

Index(['countryname', 'langname_x', 'id_x', 'corr_logcount_response_mean_x',
       'corr_logcount_response_sd_x', 'corr_logcount_abs_dist_midpoint_x',
       'langname_y', 'id_y', 'corr_logcount_response_mean_y',
       'corr_logcount_response_sd_y', 'corr_logcount_abs_dist_midpoint_y'],
      dtype='object')

In [72]:
both = both[['countryname', 'langname_x','langname_y', 'id_x','id_y',  'corr_logcount_response_mean_x','corr_logcount_response_mean_y',
       'corr_logcount_response_sd_x','corr_logcount_response_sd_y', 'corr_logcount_abs_dist_midpoint_x','corr_logcount_abs_dist_midpoint_y']]
both

,countryname,langname_x,langname_y,id_x,id_y,corr_logcount_response_mean_x,corr_logcount_response_mean_y,corr_logcount_response_sd_x,corr_logcount_response_sd_y,corr_logcount_abs_dist_midpoint_x,corr_logcount_abs_dist_midpoint_y
0,Brazil,Brazilian Portuguese,Brazilian Portuguese,txu.059173018640743,txu.059173018640743,0.162436,0.162436,-0.095865,-0.095865,0.060763,0.058824
1,Bulgaria,Bulgarian,Bulgarian,mdp.39015058560494,mdp.39015058560494,0.291458,0.291458,-0.510940,-0.510940,0.256754,0.265190
2,Chile,Spanish,Latin American Spanish,umn.31951d01452389x,mdp.39015050181174,0.199989,0.371436,-0.153504,-0.362343,0.284414,0.241720
3,China,Mandarin Chinese,Mandarin Chinese,uc1.l0088545033,uc1.l0088545033,0.191494,0.191494,-0.404744,-0.404744,0.107092,0.116186
4,Colombia,Spanish,Latin American Spanish,umn.31951d01452389x,mdp.39015050181174,0.029407,0.236561,-0.270589,-0.175261,-0.066229,-0.137527
5,Croatia,Serbian-Croatian-Bosnian,Serbian-Croatian-Bosnian,mdp.39015012891506,mdp.39015012891506,-0.363570,-0.363570,-0.037434,-0.037434,-0.237194,-0.247292
6,Curacao,Dutch,Dutch,uc1.31822004083036,uc1.31822004083036,-0.227172,-0.227172,-0.012792,-0.012792,-0.209157,-0.211604
7,Denmark,Danish,Danish,uc1.b3832576,uc1.b3832576,-0.322866,-0.322866,-0.191735,-0.191735,0.325635,0.313704
8,Egypt,Arabic,Arabic,mdp.39015043036436,mdp.39015043036436,-0.114331,-0.114331,0.133291,0.133291,-0.347282,-0.338161
9,Finland,Finnish,Finnish,mdp.39015059174667,mdp.39015059174667,0.072036,0.072036,-0.299668,-0.299668,-0.090822,-0.086969


In [73]:
both.to_csv(Path("~/Projects/hypocognition/data/processed/corr_comparison.csv").expanduser())

In [218]:
merged.columns

Index(['langname', 'glottocode', 'word', 'count', 'id', 'year', 'countryname',
       'response_mean', 'std'],
      dtype='object')

In [176]:
import plotly.graph_objects as go

df_plot = result.copy()
df_plot = df_plot.sort_values('langname')
df_plot["sample"] = df_plot["countryname"] + " – " + df_plot["langname"]


In [177]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_mean"],
    mode="markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_sd"],
    mode="markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color= 'blue')
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red")
))

fig.update_layout(
    title="Correlations between Log Count and Response Measures",
    yaxis_title="Pearson correlation",
    xaxis_title="Sample (Country – Language)",
    xaxis_tickangle=45,
    height=600,
    legend_title="Correlation type",
    template="simple_white"
)

fig.show()


In [178]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_mean"],
    mode="lines+markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime"),
    line=dict(width=2, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_response_sd"],
    mode="lines+markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color="blue"),
    line=dict(width=2, color="blue")
))

fig.add_trace(go.Scatter(
    x=df_plot["sample"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="lines+markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red"),
    line=dict(width=2, color="red")
))

fig.add_hline(
    y=0,
    line_width=2,
    line_color="black"
)


fig.update_layout(
    title="Correlations between Log Count and Response Measures",
    yaxis_title="Pearson correlation",
    xaxis_title="Sample (Country – Language)",
    xaxis_tickangle=45,
    height=700,
    legend_title="Correlation type",
    template="simple_white"
)

fig.show()


In [2130]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_response_mean"],
    mode="markers",
    name="Corr(log count, response mean)",
    marker=dict(size=9, color="lime")
))

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_response_sd"],
    mode="markers",
    name="Corr(log count, response SD)",
    marker=dict(size=9, color="blue")
))

fig.add_trace(go.Scatter(
    x=df_plot["langname"],
    y=df_plot["corr_logcount_abs_dist_midpoint"],
    mode="markers",
    name="Corr(log count, |mean − midpoint|)",
    marker=dict(size=9, color="red")
))

# horizontal zero line
fig.add_hline(
    y=0,
    line_width=1,
    line_dash="solid",
    line_color="black"
)

fig.update_layout(
    title="Correlations by Language, Searching for Hypocognition",
    xaxis_title="Language",
    yaxis_title="Pearson correlation",
    xaxis_tickangle=45,
    height=700,
    template="simple_white",
    legend_title="Correlation type"
)

fig.show()


In [ ]:
fig.write_html(Path("reference/lang_names.csv"))